# PhoBERT NER — chạy lại trên tài khoản mới
314 Train / 55 Validation, dữ liệu và mô hình không đổi. Lượt trước mất runtime sau log epoch 10; đây là lượt chạy mới, không phải resume.
Worker chạy nền. Giao diện theo dõi không giữ kernel bận; hỗ trợ sao lưu checkpoint về máy trong khi train.


In [ ]:
import os,sys,subprocess,torch
os.environ['USE_TF']='0';os.environ['TOKENIZERS_PARALLELISM']='false'
assert torch.cuda.is_available(), 'Chưa được cấp GPU; không tự train CPU.'
print('ALLOCATED_GPU',torch.cuda.get_device_name(0),torch.__version__)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','tokenizers>=0.21,<0.22','pyvi==0.1.1','pytest>=8,<10','huggingface_hub'],check=True)


In [ ]:
#@title Khôi phục mã nguồn và dữ liệu đã kiểm tra
from pathlib import Path
import base64,hashlib,io,json,zipfile
workspace=Path('/content/legal_ner_all369');workspace.mkdir(parents=True,exist_ok=True)
raw=base64.b64decode(
'...'
)
assert hashlib.sha256(raw).hexdigest()=='13629a93823051971ce193a140a67209ab6deb233e46c1ea978d17fa72bd4247'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 assert z.testzip() is None
 assert all((workspace/n).resolve().is_relative_to(workspace.resolve()) for n in z.namelist())
 z.extractall(workspace)
root=workspace/'Legal_Entity_NER'
print('FROZEN_INPUTS_RESTORED',root)


In [ ]:
# Chạy worker; kết nối trình duyệt mất không tự làm mất log/checkpoint.
run='phobert_span_all369_v1';report=root/'reports'/run;report.mkdir(parents=True,exist_ok=True)
assert not (report/'worker_status.json').exists(), 'Đã có worker, đọc trạng thái thay vì chạy trùng.'
with (report/'worker.log').open('w') as log:
 process=subprocess.Popen([sys.executable,'-u','scripts/run_colab_all.py'],cwd=root,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('WORKER_STARTED',process.pid)
from google.colab import output
from IPython.display import JSON,Javascript,display
allowed={'trained_zip':workspace/'Legal_Entity_NER_All369_Trained.zip','results':report/'RESULTS.json','history':report/'history.json'}
def transfer(key,offset=0,limit=4*1024*1024):
 path=allowed[key];offset=int(offset);limit=min(int(limit),4*1024*1024)
 assert offset>=0 and limit>0
 with path.open('rb') as stream:stream.seek(offset);block=stream.read(limit)
 return JSON({'offset':offset,'total':path.stat().st_size,'bytes':len(block),'base64':base64.b64encode(block).decode()})
def poll():
 p=report/'worker_status.json';s=json.loads(p.read_text()) if p.exists() else {'phase':'STARTING'}
 for name in ['progress.json','history.json']:
  try:
   if (report/name).exists():s[name]=json.loads((report/name).read_text())
  except ValueError:pass
 p=report/'train.log' if (report/'train.log').exists() else report/'worker.log'
 if p.exists():
  with p.open('rb') as f:f.seek(max(0,p.stat().st_size-4000));s['log_tail']=f.read().decode(errors='replace')
 return JSON(s)
output.register_callback('ner_all369.status',poll);output.register_callback('ner_all369.transfer',transfer)

# Bản sao chỉ chứa checkpoint thực đã được worker lưu; không dùng làm kết quả cuối.
def checkpoint_snapshot():
 import os,shutil,hashlib,zipfile
 folder=root/'checkpoints'/run
 marker=folder/'training_state.json'
 if not marker.exists() or not (folder/'model.pt').exists():
  return JSON({'available':False,'reason':'NO_CHECKPOINT_YET'})
 epoch=json.loads(marker.read_text())['epoch']
 output_file=workspace/'NER_All369_Checkpoint_Backup.zip'
 signature=(folder/'model.pt').stat().st_mtime_ns
 if not output_file.exists() or getattr(checkpoint_snapshot,'signature',None)!=signature:
  temporary=output_file.with_suffix('.zip.tmp')
  source_paths=[folder/n for n in ['model.pt','config.json','training_state.json','decoder_policy.json','inference_settings.json'] if (folder/n).exists()]
  source_paths += [report/n for n in ['history.json','EXPERIMENT.json','PRETRAINING_ORIGIN.json'] if (report/n).exists()]
  metadata={'kind':'INTERMEDIATE_CHECKPOINT_NOT_FINAL_RESULTS','epoch':epoch,'run_name':run,'train_documents':314,'validation_documents':55,'input_sha256':'13629a93823051971ce193a140a67209ab6deb233e46c1ea978d17fa72bd4247','model_sha256':hashlib.file_digest((folder/'model.pt').open('rb'),'sha256').hexdigest()}
  with zipfile.ZipFile(temporary,'w',zipfile.ZIP_STORED) as z:
   for path in source_paths:z.write(path,str(path.relative_to(root)))
   z.writestr('BACKUP_MANIFEST.json',json.dumps(metadata))
  assert (folder/'model.pt').stat().st_mtime_ns==signature,'Checkpoint changed; retry snapshot'
  temporary.replace(output_file)
  checkpoint_snapshot.signature=signature
  checkpoint_snapshot.digest=hashlib.file_digest(output_file.open('rb'),'sha256').hexdigest()
 allowed['checkpoint_backup']=output_file
 return JSON({'available':True,'epoch':epoch,'bytes':output_file.stat().st_size,'sha256':checkpoint_snapshot.digest})
output.register_callback('ner_all369.snapshot',checkpoint_snapshot)

display(Javascript("window.nckhSnapshot=()=>google.colab.kernel.invokeFunction('ner_all369.snapshot',[],{});window.nckhPoll=()=>google.colab.kernel.invokeFunction('ner_all369.status',[],{});window.nckhTransfer=(key,offset,limit)=>google.colab.kernel.invokeFunction('ner_all369.transfer',[key,offset,limit],{});document.body.appendChild(Object.assign(document.createElement('pre'),{textContent:'NER_ALL369_WORKER_MONITOR_READY'}));"))


In [ ]:
# Theo dõi bằng callback để kernel rảnh cho việc đọc trạng thái và sao lưu.
from IPython.display import Javascript,display
display(Javascript("""
const box=document.createElement('pre');document.body.appendChild(box);
let running=false;
async function update(){
 if(running)return;running=true;
 try{
  const response=await google.colab.kernel.invokeFunction('ner_all369.status',[],{});
  let state=response.data['application/json'];if(typeof state==='string')state=JSON.parse(state);
  box.textContent=JSON.stringify(state,null,2);
  if(['COMPLETE','FAILED'].includes(state.phase))clearInterval(window.nerWatch);
 }catch(e){box.textContent='Chưa đọc được trạng thái: '+e;}
 finally{running=false;}
}
window.nerWatch=setInterval(update,30000);update();
"""))


In [ ]:
state=json.loads((report/'worker_status.json').read_text());assert state['phase']=='COMPLETE',state
from google.colab import files
files.download(state['artifact']['path'])
